# 22 - llama.cpp baseline on the T4 (CUDA build for sm_75, f16 GGUF)
Run on a **GPU T4 x2** session with Internet on and Persistence = Variables and Files. Press **Run All**; every stage skips work that is already done, so after a crash you can simply press Run All again.

Order: setup, build llama-server (10-20 minutes the first time), download the f16 GGUF / tokenizer / ShareGPT / benchmark client, start the server (16 slots), ShareGPT sweep (c=1 / c=4 / c=16,32,64, two repetitions each), prefill sweep (2k/4k/8k tokens, one big slot), bundle the results into one zip. Expect about 60-90 minutes in total. Progress lines appear after every repetition.

Do not use *Factory reset* (it wipes `/kaggle/working`). Close the laptop lid only after the zip is downloaded.

In [ ]:
import os, sys, re, json, time, signal, shutil, subprocess, threading, glob

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = "/kaggle/working"
LLAMA = f"{WORK}/llama.cpp"
BIN = f"{LLAMA}/build/bin/llama-server"
GGUF_DIR = f"{WORK}/gguf"
HF_MODEL = f"{WORK}/models/Qwen2.5-1.5B-Instruct"
REPO = f"{WORK}/turingserve"
SHAREGPT = f"{WORK}/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
RESULTS = f"{WORK}/results"
PORT = 8002
NP, CTX = 16, 65536      # 16 parallel slots with 4096 tokens each; requests beyond 16 in flight queue inside the server
RUN_PREFILL = True       # the 2k/4k/8k prefill sweep needs a restart with one big slot
os.makedirs(RESULTS, exist_ok=True)
os.environ["TS_SHAREGPT"] = SHAREGPT
os.environ["TS_KAGGLE"] = "1"
srv = None
GGUF = None


def gpu_used_mib():
    r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                       capture_output=True, text=True)
    return int(r.stdout.strip().splitlines()[0]) if r.returncode == 0 and r.stdout.strip() else -1


def stop_server():
    global srv
    if srv is not None and srv.poll() is None:
        try:
            os.killpg(os.getpgid(srv.pid), signal.SIGTERM)
            srv.wait(timeout=30)
        except Exception:
            try:
                os.killpg(os.getpgid(srv.pid), signal.SIGKILL)
            except Exception:
                pass
    srv = None
    subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)   # also clears a server left over from an earlier kernel
    time.sleep(3)
    for _ in range(30):
        if 0 <= gpu_used_mib() < 500:
            break
        time.sleep(2)
    print("server stopped, GPU 0 memory used:", gpu_used_mib(), "MiB")


def _launch(ctx, np_, log_path, fa):
    log = open(log_path, "w")
    cmd = [BIN, "-m", GGUF, "--alias", "qwen", "-ngl", "99", "-c", str(ctx), "-np", str(np_),
           "--port", str(PORT), "--host", "127.0.0.1", *fa]
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)


def start_server(ctx, np_, log_name):
    """Launch llama-server, wait for /health, print the first completion."""
    global srv
    import requests
    stop_server()
    log_path = f"{WORK}/{log_name}"
    for fa in (["-fa", "on"], ["-fa"]):          # newer builds take "-fa on", older ones a bare "-fa"
        srv = _launch(ctx, np_, log_path, fa)
        t0 = time.time()
        up = False
        for i in range(120):
            try:
                if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
                    up = True
                    break
            except Exception:
                pass
            if srv.poll() is not None:
                break
            time.sleep(3)
        if up:
            print(f"llama-server up after {time.time() - t0:.0f}s (ctx {ctx}, slots {np_}, flash-attn flag {fa})")
            break
        print("did not start with flags", fa, "- log tail:")
        print(open(log_path).read()[-1200:])
        stop_server()
    else:
        raise RuntimeError("llama-server did not start, log tail above")
    r = requests.post(f"http://localhost:{PORT}/v1/completions",
                      json={"model": "qwen", "prompt": "The capital of France is", "max_tokens": 16, "temperature": 0})
    print("test completion:", r.json()["choices"][0]["text"])


def run_bench(label, extra):
    """Run serve_bench, streaming its output so progress is visible; a heartbeat prints every 2 minutes."""
    cmd = [sys.executable, "-u", "-m", "benchmarks.serve_bench", "--base-url", f"http://localhost:{PORT}",
           "--engine", "llamacpp", "--model", "qwen", "--tokenizer", HF_MODEL, "--out", RESULTS, "--label", label, *extra]
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    t0 = time.time()
    done = threading.Event()

    def beat():
        while not done.wait(120):
            print(f"  ... {label} still running ({(time.time() - t0) / 60:.0f} min)", flush=True)

    threading.Thread(target=beat, daemon=True).start()
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Token indices sequence length" in line or "Running this sequence" in line:
            continue
        print(line, end="", flush=True)
    p.wait()
    done.set()
    print(f"[{label}] finished in {(time.time() - t0) / 60:.1f} min, exit code {p.returncode}\n")
    return p.returncode


def result_exists(label):
    return bool(glob.glob(f"{RESULTS}/*_llamacpp_*_{label}.json"))


print("setup ok; GPU 0 memory used:", gpu_used_mib(), "MiB")

## 1. Build llama-server
The build log is condensed to one line per 5%. If cmake cannot find CUDA the error and log tail are printed.

In [ ]:
if not os.path.exists(BIN):
    if not os.path.isdir(f"{LLAMA}/.git"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)
    nvcc = shutil.which("nvcc") or ("/usr/local/cuda/bin/nvcc" if os.path.exists("/usr/local/cuda/bin/nvcc") else None)
    cfg = ["cmake", "-S", LLAMA, "-B", f"{LLAMA}/build", "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
           "-DCMAKE_BUILD_TYPE=Release"]
    if nvcc:
        cfg.append(f"-DCMAKE_CUDA_COMPILER={nvcc}")
    # The Kaggle image has no libcuda.so link for cmake to find (CUDA::cuda_driver), because the driver library is injected
    # into the container at runtime. Link a directory to the real driver library and point cmake at it.
    stub_dir = f"{WORK}/cudastub"
    os.makedirs(stub_dir, exist_ok=True)
    ld = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout.splitlines()
    drv = [l.split("=>")[-1].strip() for l in ld if "libcuda.so.1" in l and "x86-64" in l]
    if not drv:
        raise RuntimeError("could not find the NVIDIA driver library libcuda.so.1 (is a GPU attached to this session?)")
    link = f"{stub_dir}/libcuda.so"
    if os.path.lexists(link):
        os.remove(link)
    os.symlink(drv[0], link)
    print("driver library:", drv[0])
    cfg += [f"-DCMAKE_LIBRARY_PATH={stub_dir}:/usr/local/cuda/lib64/stubs", "-DCUDAToolkit_ROOT=/usr/local/cuda", "--fresh"]
    r = subprocess.run(cfg, capture_output=True, text=True)
    if r.returncode != 0:
        err = (r.stdout + r.stderr)
        i = err.find("CMake Error")
        print(err[i:i + 2500] if i >= 0 else err[-2500:])
        raise RuntimeError("cmake configure failed")
    # the CUDA build takes 10-20 minutes; print the percentage every 5%
    p = subprocess.Popen(["cmake", "--build", f"{LLAMA}/build", "--config", "Release", "-j", "4", "--target", "llama-server"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    last, tail = -5, []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        m = re.match(r"\[\s*(\d+)%\]", line)
        if m and int(m.group(1)) >= last + 5:
            last = int(m.group(1))
            print(f"build {last}%", flush=True)
    p.wait()
    if p.returncode != 0 or not os.path.exists(BIN):
        print("".join(tail))
        raise RuntimeError("llama.cpp build failed, log tail above")
print("llama-server binary:", BIN)
v = subprocess.run([BIN, "--version"], capture_output=True, text=True)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=LLAMA, capture_output=True, text=True).stdout.strip()
info = f"llama.cpp commit {commit}\n{(v.stdout + v.stderr).strip()[-600:]}\n"
print(info)
open(f"{WORK}/llama_versions.txt", "w").write(info)

## 2. Models, dataset and benchmark client
The benchmark client is the same `serve_bench` at tag `m3-rc` that produced the TuringServe numbers.

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

# f16 GGUF so llama.cpp runs the same precision as TuringServe and vLLM
found = glob.glob(f"{GGUF_DIR}/*f16*.gguf")
if not found:
    try:
        snapshot_download("bartowski/Qwen2.5-1.5B-Instruct-GGUF", allow_patterns=["*f16.gguf"], local_dir=GGUF_DIR)
    except Exception as e:
        print("download failed:", e)
    found = glob.glob(f"{GGUF_DIR}/*f16*.gguf")

if not os.path.exists(f"{HF_MODEL}/model.safetensors"):      # also used as the tokenizer by the benchmark client
    snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", local_dir=HF_MODEL)

if not found:
    print("no f16 GGUF in that repo, converting from the HF checkpoint")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf", "sentencepiece"], capture_output=True)
    os.makedirs(GGUF_DIR, exist_ok=True)
    out = f"{GGUF_DIR}/qwen2.5-1.5b-f16.gguf"
    subprocess.run([sys.executable, f"{LLAMA}/convert_hf_to_gguf.py", HF_MODEL, "--outtype", "f16", "--outfile", out], check=True)
    found = [out]
GGUF = found[0]
print("GGUF:", GGUF, f"({os.path.getsize(GGUF) / 1e9:.2f} GB)")

if not os.path.exists(SHAREGPT):
    hf_hub_download("anon8231489123/ShareGPT_Vicuna_unfiltered", "ShareGPT_V3_unfiltered_cleaned_split.json",
                    repo_type="dataset", local_dir=f"{WORK}/datasets/sharegpt")
print("dataset ok:", os.path.exists(SHAREGPT))

if not os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "clone", "https://github.com/Muskanpandey0304/turingserve.git", REPO], check=True)
subprocess.run(["git", "fetch", "--tags", "--force"], cwd=REPO, check=True, capture_output=True)
subprocess.run(["git", "checkout", "-f", "m3-rc"], cwd=REPO, check=True, capture_output=True)
print("repo at:", subprocess.run(["git", "log", "-1", "--oneline"], cwd=REPO, capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "datasets", "transformers"], capture_output=True)

## 3. Start llama-server (port 8002)
If it fails to start, the log tail is printed. Paste it and stop.

In [ ]:
start_server(CTX, NP, "llama.log")
print("--- offload / flash-attention lines from the llama.cpp log (record these in docs/baselines.md) ---")
for line in open(f"{WORK}/llama.log"):
    if any(k in line.lower() for k in ("offloaded", "flash", "cuda", "n_slots", "n_ctx", "compute capability")):
        print(line.rstrip()[:200])

## 4. ShareGPT sweep
With 16 slots, levels 32 and 64 queue extra requests inside the server; that is a property of this configuration and goes in the notes.

In [ ]:
# c=1 is the slow level (requests run one at a time), so it uses fewer prompts; the high levels keep large samples.
STAGES = [
    ("c1",  ["--num-prompts", "60",  "--concurrency", "1"]),
    ("c4",  ["--num-prompts", "150", "--concurrency", "4"]),
    ("hi",  ["--num-prompts", "400", "--concurrency", "16,32,64"]),
]
COMMON = ["--dataset", "sharegpt", "--mode", "concurrency", "--repeat", "2", "--ignore-eos", "--gpu-log"]
failed = []
for name, extra in STAGES:
    label = f"t4-1.5b-{name}"
    if result_exists(label):
        print(f"[{label}] result already exists, skipping")
        continue
    if run_bench(label, [*COMMON, *extra]) != 0:
        failed.append(label)
print("stages that failed:", failed or "none")

## 5. Prefill sweep (restarts the server with one 16k-token slot)

In [ ]:
label = "t4-1.5b-prefill-cold"
if not RUN_PREFILL:
    print("prefill skipped (RUN_PREFILL = False)")
elif result_exists(label):
    print(f"[{label}] result already exists, skipping")
else:
    start_server(16384, 1, "llama_prefill.log")
    run_bench(label, ["--dataset", "prefill", "--num-prompts", "500", "--mode", "concurrency",
                      "--concurrency", "1", "--repeat", "3", "--gpu-log"])

## 6. Summary and download
Download **`/kaggle/working/llamacpp_bundle.zip`** (results, logs and versions in one file), then stop the session.

In [ ]:
stop_server()

print(f"{'file':58}{'c':>4}{'ok/n':>9}{'err':>5}{'tok/s':>9}{'ttft p50 ms':>13}{'tpot p50 ms':>13}")
for f in sorted(glob.glob(f"{RESULTS}/*_llamacpp_*.json")):
    d = json.load(open(f))
    for r in d["runs"]:
        m = r["median"]
        p50 = lambda t: (t.get("p50") or 0) * 1000
        print(f"{os.path.basename(f)[:57]:58}{r['level']:>4}{m['num_ok']:>5}/{m['num_requests']:<3}{m['errors']:>5}"
              f"{m['output_throughput_tok_s']:>9.1f}{p50(m['ttft']):>13.1f}{p50(m['tpot']):>13.1f}")

print("\nprefill (cold) per prompt length:")
cold_ok = True
for f in sorted(glob.glob(f"{RESULTS}/*_llamacpp_prefill_*.json")):
    r = json.load(open(f))["runs"][0]
    by = {}
    for q in r["requests"]:
        if q.get("first") and not q.get("error"):
            by.setdefault(q["prompt_tokens"], []).append((q["first"] - q["send"]) * 1000)
    print(f"{'prompt tokens':>14}{'n':>4}{'TTFT median ms':>16}{'min':>8}{'max':>8}{'prefill tok/s':>15}")
    for L in sorted(by):
        v = sorted(by[L])
        med = v[len(v) // 2]
        print(f"{L:>14}{len(v):>4}{med:>16.0f}{v[0]:>8.0f}{v[-1]:>8.0f}{L / (med / 1000):>15.0f}")
        if L >= 7000 and med < 300:
            cold_ok = False
print("sanity check passed: 8k-token TTFT is real cold prefill" if cold_ok else
      "WARNING: an 8k-token prompt came back in under 300 ms, which a T4 cannot do cold; the prompt cache is probably hitting")

bundle = f"{WORK}/llamacpp_bundle"
shutil.rmtree(bundle, ignore_errors=True)
os.makedirs(bundle)
for f in glob.glob(f"{RESULTS}/*_llamacpp_*.json") + glob.glob(f"{WORK}/llama*.log") + [f"{WORK}/llama_versions.txt"]:
    if os.path.exists(f):
        shutil.copy(f, bundle)
shutil.make_archive(f"{WORK}/llamacpp_bundle", "zip", bundle)
print("\nDownload this one file:", f"{WORK}/llamacpp_bundle.zip", f"({os.path.getsize(WORK + '/llamacpp_bundle.zip') / 1e6:.1f} MB)")